In [1]:
!pip install -q \
    "transformers>=4.48,<5" \
    "sentencepiece>=0.2" \
    "safetensors>=0.4" \
    "accelerate>=1.2"

print("Stage 4 libraries installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Stage 4 libraries installed.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import json
import re
from collections import Counter

import pandas as pd
import torch

STAGE3_JSON = Path(
    "/content/drive/MyDrive/KidStory-Qwen2.5/"
    "learnguard_stage3_results/"
    "20260728_105009_"
    "honesty_when_a_child_finds_a_lost_wallet_"
    "question_candidates.json"
)

if not STAGE3_JSON.exists():
    raise FileNotFoundError(
        f"Stage 3 file was not found: {STAGE3_JSON}"
    )

with STAGE3_JSON.open(
    "r",
    encoding="utf-8",
) as file:
    stage3_record = json.load(file)

if stage3_record["stage3_status"] != "COMPLETE":
    raise ValueError(
        "Stage 3 has not been completed."
    )

TOPIC = stage3_record["topic"]
AGE_GROUP = stage3_record["age_group"]
STORY = stage3_record["story"]

QUESTION_CANDIDATES = stage3_record[
    "question_candidates"
]

print("Stage 3 result loaded successfully.")
print("Topic:", TOPIC)
print("Age group:", AGE_GROUP)
print(
    "Question candidates:",
    len(QUESTION_CANDIDATES),
)

print("\nFIRST CANDIDATE")
print(
    "ID:",
    QUESTION_CANDIDATES[0]["candidate_id"],
)
print(
    "Question:",
    QUESTION_CANDIDATES[0]["question"],
)
print(
    "Intended answer:",
    QUESTION_CANDIDATES[0]["answer_span"],
)

Stage 3 result loaded successfully.
Topic: honesty when a child finds a lost wallet
Age group: 8–10
Question candidates: 17

FIRST CANDIDATE
ID: Q001
Question: Who was the owner of the wallet?
Intended answer: Emily


In [3]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline,
)

QA_MODEL_PATH = Path(
    "/content/drive/MyDrive/QA_Fairytale/"
    "QA_Genera_New/"
    "t5base_fairytaleQA_tagged_e20_bs64_beam8/"
    "checkpoint-1250_polish/"
    "checkpoint-400_final"
)

required_qa_files = [
    "config.json",
    "model.safetensors",
    "tokenizer_config.json",
]

if not QA_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"QA model was not found: {QA_MODEL_PATH}"
    )

missing_qa_files = [
    filename
    for filename in required_qa_files
    if not (
        QA_MODEL_PATH / filename
    ).exists()
]

if missing_qa_files:
    raise FileNotFoundError(
        f"Missing QA model files: {missing_qa_files}"
    )

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Loading the fine-tuned QA model...")

qa_tokenizer = AutoTokenizer.from_pretrained(
    QA_MODEL_PATH
)

qa_model = (
    AutoModelForSeq2SeqLM.from_pretrained(
        QA_MODEL_PATH
    )
    .to(device)
    .eval()
)

print("Fine-tuned QA model loaded.")
print("Device:", device)
print(
    "Model class:",
    qa_model.__class__.__name__,
)

Loading the fine-tuned QA model...
Fine-tuned QA model loaded.
Device: cuda
Model class: T5ForConditionalGeneration


In [4]:
print(
    "Loading independent extractive QA verifier..."
)

extractive_qa = pipeline(
    task="question-answering",
    model="deepset/roberta-base-squad2",
    tokenizer="deepset/roberta-base-squad2",
    device=0 if device == "cuda" else -1,
)

print(
    "Independent extractive QA verifier loaded."
)

Loading independent extractive QA verifier...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Device set to use cuda:0


Independent extractive QA verifier loaded.


In [5]:
def normalize_text(text: str) -> str:
    """Normalize text before comparing answers."""

    text = (text or "").lower().strip()
    text = re.sub(
        r"[^a-z0-9 ?!]+",
        " ",
        text,
    )
    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


def token_f1(
    predicted_answer: str,
    intended_answer: str,
) -> float:
    """
    Measure word overlap between two answers.

    1.0 means perfect token agreement.
    0.0 means no matching answer tokens.
    """

    predicted_tokens = normalize_text(
        predicted_answer
    ).split()

    intended_tokens = normalize_text(
        intended_answer
    ).split()

    if (
        len(predicted_tokens) == 0
        or len(intended_tokens) == 0
    ):
        return float(
            predicted_tokens == intended_tokens
        )

    predicted_counts = Counter(
        predicted_tokens
    )
    intended_counts = Counter(
        intended_tokens
    )

    shared_count = sum(
        min(
            predicted_counts[token],
            intended_counts[token],
        )
        for token in set(intended_counts)
    )

    if shared_count == 0:
        return 0.0

    precision = (
        shared_count
        / len(predicted_tokens)
    )

    recall = (
        shared_count
        / len(intended_tokens)
    )

    return (
        2
        * precision
        * recall
        / (precision + recall)
    )


def make_qa_input(
    story: str,
    question: str,
) -> str:
    """Format input exactly for the fine-tuned QA model."""

    return (
        "Answer concisely and factually "
        "based on the story.\n"
        "<story>\n"
        + story.strip()
        + "\n</story>\n"
        + "<question>\n"
        + question.strip()
        + "\n</question>\n"
        + "<answer>\n"
    )


@torch.inference_mode()
def generate_qa_answer(
    story: str,
    question: str,
    max_new_tokens: int = 64,
) -> str:
    """Answer a question using your fine-tuned T5 QA model."""

    model_input = make_qa_input(
        story,
        question,
    )

    encoded_input = qa_tokenizer(
        model_input,
        return_tensors="pt",
        truncation=True,
        max_length=768,
    ).to(device)

    generated_output = qa_model.generate(
        **encoded_input,
        num_beams=1,
        do_sample=False,
        max_new_tokens=max_new_tokens,
    )

    answer = qa_tokenizer.decode(
        generated_output[0],
        skip_special_tokens=True,
    ).strip()

    return answer


def generate_extractive_answer(
    story: str,
    question: str,
) -> tuple[str, float]:
    """Answer using the independent extractive verifier."""

    output = extractive_qa(
        question=question,
        context=story,
    )

    answer = output.get(
        "answer",
        "",
    ).strip()

    confidence = float(
        output.get(
            "score",
            0.0,
        )
    )

    return answer, confidence


print("QA verification functions are ready.")

QA verification functions are ready.


In [6]:
test_candidate = QUESTION_CANDIDATES[0]

test_question = test_candidate["question"]
test_intended_answer = test_candidate[
    "answer_span"
]

test_generative_answer = generate_qa_answer(
    story=STORY,
    question=test_question,
)

(
    test_extractive_answer,
    test_extractive_confidence,
) = generate_extractive_answer(
    story=STORY,
    question=test_question,
)

test_f1_generative = token_f1(
    test_generative_answer,
    test_intended_answer,
)

test_f1_extractive = token_f1(
    test_extractive_answer,
    test_intended_answer,
)

test_model_agreement = token_f1(
    test_generative_answer,
    test_extractive_answer,
)

print("=" * 70)
print("STAGE 4 SINGLE-CANDIDATE TEST")
print("=" * 70)

print(
    "Candidate ID:",
    test_candidate["candidate_id"],
)
print("Question:", test_question)
print(
    "Intended answer:",
    test_intended_answer,
)
print(
    "Fine-tuned QA answer:",
    test_generative_answer,
)
print(
    "Extractive QA answer:",
    test_extractive_answer,
)
print(
    "Extractive confidence:",
    round(test_extractive_confidence, 4),
)
print(
    "F1: fine-tuned QA vs intended:",
    round(test_f1_generative, 4),
)
print(
    "F1: extractive QA vs intended:",
    round(test_f1_extractive, 4),
)
print(
    "F1: agreement between QA models:",
    round(test_model_agreement, 4),
)

The following generation flags are not valid and may be ignored: ['length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


STAGE 4 SINGLE-CANDIDATE TEST
Candidate ID: Q001
Question: Who was the owner of the wallet?
Intended answer: Emily
Fine-tuned QA answer: Mr. Johnson.
Extractive QA answer: Mr. Johnson
Extractive confidence: 0.4695
F1: fine-tuned QA vs intended: 0.0
F1: extractive QA vs intended: 0.0
F1: agreement between QA models: 1.0


In [7]:
def assign_qc_level(
    f1_generative: float,
    f1_extractive: float,
    f1_agreement: float,
    extractive_confidence: float,
) -> str:
    """
    Assign strict, medium, mild or reject status.

    All important conditions must pass at the same level.
    """

    qc_thresholds = [
        {
            "level": "strict",
            "f1_generative": 0.85,
            "f1_extractive": 0.50,
            "f1_agreement": 0.50,
            "extractive_confidence": 0.20,
        },
        {
            "level": "medium",
            "f1_generative": 0.85,
            "f1_extractive": 0.45,
            "f1_agreement": 0.45,
            "extractive_confidence": 0.15,
        },
        {
            "level": "mild",
            "f1_generative": 0.80,
            "f1_extractive": 0.40,
            "f1_agreement": 0.40,
            "extractive_confidence": 0.10,
        },
    ]

    for thresholds in qc_thresholds:

        passed = (
            f1_generative
            >= thresholds["f1_generative"]
            and f1_extractive
            >= thresholds["f1_extractive"]
            and f1_agreement
            >= thresholds["f1_agreement"]
            and extractive_confidence
            >= thresholds["extractive_confidence"]
        )

        if passed:
            return thresholds["level"]

    return "reject"


test_qc_level = assign_qc_level(
    f1_generative=test_f1_generative,
    f1_extractive=test_f1_extractive,
    f1_agreement=test_model_agreement,
    extractive_confidence=(
        test_extractive_confidence
    ),
)

print("Test candidate QC level:", test_qc_level)

Test candidate QC level: reject


In [8]:
def verify_question_candidate(
    candidate: dict,
) -> dict:
    """Run both QA verifiers and assign a QC level."""

    question = candidate["question"]
    intended_answer = candidate[
        "answer_span"
    ]

    generative_answer = generate_qa_answer(
        story=STORY,
        question=question,
    )

    (
        extractive_answer,
        extractive_confidence,
    ) = generate_extractive_answer(
        story=STORY,
        question=question,
    )

    f1_generative = token_f1(
        generative_answer,
        intended_answer,
    )

    f1_extractive = token_f1(
        extractive_answer,
        intended_answer,
    )

    f1_agreement = token_f1(
        generative_answer,
        extractive_answer,
    )

    qc_level = assign_qc_level(
        f1_generative=f1_generative,
        f1_extractive=f1_extractive,
        f1_agreement=f1_agreement,
        extractive_confidence=(
            extractive_confidence
        ),
    )

    result = candidate.copy()

    result.update(
        {
            "generative_qa_answer": (
                generative_answer
            ),
            "extractive_qa_answer": (
                extractive_answer
            ),
            "extractive_confidence": round(
                extractive_confidence,
                4,
            ),
            "f1_generative": round(
                f1_generative,
                4,
            ),
            "f1_extractive": round(
                f1_extractive,
                4,
            ),
            "f1_agreement": round(
                f1_agreement,
                4,
            ),
            "qc_level": qc_level,
            "approved": (
                qc_level != "reject"
            ),
        }
    )

    return result


stage4_results = []

for candidate_number, candidate in enumerate(
    QUESTION_CANDIDATES,
    start=1,
):
    print(
        f"Verifying candidate "
        f"{candidate_number}/"
        f"{len(QUESTION_CANDIDATES)}: "
        f"{candidate['candidate_id']}"
    )

    verification_result = (
        verify_question_candidate(
            candidate
        )
    )

    stage4_results.append(
        verification_result
    )

    print(
        "QC level:",
        verification_result["qc_level"],
    )

print("\nAll candidate questions were verified.")

Verifying candidate 1/17: Q001
QC level: reject
Verifying candidate 2/17: Q002
QC level: reject
Verifying candidate 3/17: Q003
QC level: reject
Verifying candidate 4/17: Q004
QC level: strict
Verifying candidate 5/17: Q005
QC level: reject
Verifying candidate 6/17: Q006
QC level: reject
Verifying candidate 7/17: Q007
QC level: reject
Verifying candidate 8/17: Q008
QC level: reject
Verifying candidate 9/17: Q009


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


QC level: reject
Verifying candidate 10/17: Q010
QC level: reject
Verifying candidate 11/17: Q011
QC level: reject
Verifying candidate 12/17: Q012
QC level: reject
Verifying candidate 13/17: Q013
QC level: reject
Verifying candidate 14/17: Q014
QC level: strict
Verifying candidate 15/17: Q015
QC level: reject
Verifying candidate 16/17: Q016
QC level: reject
Verifying candidate 17/17: Q017
QC level: reject

All candidate questions were verified.


In [9]:
stage4_results_df = pd.DataFrame(
    stage4_results
)

pd.set_option(
    "display.max_colwidth",
    None,
)

print("QC LEVEL DISTRIBUTION")

print(
    stage4_results_df[
        "qc_level"
    ].value_counts()
)

approved_results_df = stage4_results_df[
    stage4_results_df["approved"]
].copy()

print(
    "\nApproved QA pairs:",
    len(approved_results_df),
)

print(
    "Approved unique answer spans:",
    approved_results_df[
        "answer_span"
    ].nunique(),
)

display(
    stage4_results_df[
        [
            "candidate_id",
            "question",
            "answer_span",
            "generative_qa_answer",
            "extractive_qa_answer",
            "extractive_confidence",
            "f1_generative",
            "f1_extractive",
            "f1_agreement",
            "qc_level",
        ]
    ]
)

QC LEVEL DISTRIBUTION
qc_level
reject    15
strict     2
Name: count, dtype: int64

Approved QA pairs: 2
Approved unique answer spans: 2


,candidate_id,question,answer_span,generative_qa_answer,extractive_qa_answer,extractive_confidence,f1_generative,f1_extractive,f1_agreement,qc_level
0,Q001,Who was the owner of the wallet?,Emily,Mr. Johnson.,Mr. Johnson,0.4695,0.0000,0.0000,1.0000,reject
1,Q002,Who was the owner of the lost wallet?,Emily,Mr. Johnson.,Mr. Johnson,0.6055,0.0000,0.0000,1.0000,reject
2,Q003,Where did Emily and Timmy meet Mr. Johnson?,the park,the nearby community centre .,community centre,0.4044,0.3333,0.0000,0.6667,reject
3,Q004,Where did the man walk through earlier that day?,the park,the park .,the park,0.4009,1.0000,1.0000,1.0000,strict
4,Q005,Where did Emily and Timmy meet to find the owner of the wallet?,the park,at the nearby community centre .,community centre,0.4275,0.2857,0.0000,0.5714,reject
5,Q006,Who did Emily ask to help her find the owner of the wallet?,Timmy,her friend Timmy.,Timmy,0.5408,0.5000,1.0000,0.5000,reject
6,Q007,What did Emily find on the ground?,a small leather wallet,"a small leather wallet. Inside were some money, a business card and an identification card.",a small leather wallet,0.4663,0.4211,1.0000,0.4211,reject
7,Q008,What did Emily discover on the ground?,a small leather wallet,"a small leather wallet. Inside were some money, a business card and an identification card.",a small leather wallet,0.5151,0.4211,1.0000,0.4211,reject
8,Q009,What did Emily find on a sunny day in the park?,a small leather wallet,"a small leather wallet. Inside were some money, a business card and an identification card.",a small leather wallet,0.4743,0.4211,1.0000,0.4211,reject
9,Q010,What did Emily and Timmy find on the business card?,the number,a telephone number and an identification card .,a telephone number,0.5996,0.2222,0.4000,0.6000,reject


In [10]:
def normalize_answer_for_scoring(
    answer: str,
) -> str:
    """
    Apply SQuAD-style normalization:
    lowercase, remove punctuation and remove articles.
    """

    answer = (answer or "").lower()

    answer = re.sub(
        r"[^a-z0-9\s]",
        " ",
        answer,
    )

    answer = re.sub(
        r"\b(a|an|the)\b",
        " ",
        answer,
    )

    answer = re.sub(
        r"\s+",
        " ",
        answer,
    ).strip()

    return answer


def normalized_token_f1(
    prediction: str,
    reference: str,
) -> float:
    """Calculate token F1 after answer normalization."""

    predicted_tokens = (
        normalize_answer_for_scoring(
            prediction
        ).split()
    )

    reference_tokens = (
        normalize_answer_for_scoring(
            reference
        ).split()
    )

    if not predicted_tokens or not reference_tokens:
        return float(
            predicted_tokens == reference_tokens
        )

    predicted_counts = Counter(
        predicted_tokens
    )

    reference_counts = Counter(
        reference_tokens
    )

    shared_count = sum(
        min(
            predicted_counts[token],
            reference_counts[token],
        )
        for token in set(reference_counts)
    )

    if shared_count == 0:
        return 0.0

    precision = (
        shared_count
        / len(predicted_tokens)
    )

    recall = (
        shared_count
        / len(reference_tokens)
    )

    return (
        2
        * precision
        * recall
        / (precision + recall)
    )


def first_answer_sentence(
    answer: str,
) -> str:
    """Retain the first answer sentence from verbose output."""

    parts = re.split(
        r"(?<=[.!?])\s+",
        (answer or "").strip(),
    )

    return parts[0].strip() if parts else ""


def contains_answer_tokens(
    prediction: str,
    reference: str,
) -> bool:
    """Check whether reference tokens occur consecutively."""

    prediction_tokens = (
        normalize_answer_for_scoring(
            prediction
        ).split()
    )

    reference_tokens = (
        normalize_answer_for_scoring(
            reference
        ).split()
    )

    if not prediction_tokens or not reference_tokens:
        return False

    reference_length = len(
        reference_tokens
    )

    for start in range(
        len(prediction_tokens)
        - reference_length
        + 1
    ):
        window = prediction_tokens[
            start:start + reference_length
        ]

        if window == reference_tokens:

            # For a one-word intended answer, permit
            # no more than two additional words.
            if (
                reference_length == 1
                and len(prediction_tokens)
                > reference_length + 2
            ):
                return False

            return True

    return False


def robust_answer_score(
    prediction: str,
    reference: str,
) -> float:
    """
    Compare the full output and its first sentence,
    then account for concise answer containment.
    """

    concise_prediction = first_answer_sentence(
        prediction
    )

    scores = [
        normalized_token_f1(
            prediction,
            reference,
        ),
        normalized_token_f1(
            concise_prediction,
            reference,
        ),
    ]

    if contains_answer_tokens(
        concise_prediction,
        reference,
    ):
        scores.append(1.0)

    return max(scores)


print("Robust answer-comparison functions are ready.")

Robust answer-comparison functions are ready.


In [11]:
stage4_robust_results = []

for original_result in stage4_results:

    intended_answer = original_result[
        "answer_span"
    ]

    generative_answer = original_result[
        "generative_qa_answer"
    ]

    extractive_answer = original_result[
        "extractive_qa_answer"
    ]

    robust_f1_generative = robust_answer_score(
        generative_answer,
        intended_answer,
    )

    robust_f1_extractive = robust_answer_score(
        extractive_answer,
        intended_answer,
    )

    robust_f1_agreement = robust_answer_score(
        generative_answer,
        extractive_answer,
    )

    robust_qc_level = assign_qc_level(
        f1_generative=robust_f1_generative,
        f1_extractive=robust_f1_extractive,
        f1_agreement=robust_f1_agreement,
        extractive_confidence=original_result[
            "extractive_confidence"
        ],
    )

    robust_result = original_result.copy()

    # Preserve the original metrics.
    robust_result.update(
        {
            "original_qc_level": original_result[
                "qc_level"
            ],
            "original_f1_generative": (
                original_result[
                    "f1_generative"
                ]
            ),
            "original_f1_extractive": (
                original_result[
                    "f1_extractive"
                ]
            ),
            "original_f1_agreement": (
                original_result[
                    "f1_agreement"
                ]
            ),
            "robust_f1_generative": round(
                robust_f1_generative,
                4,
            ),
            "robust_f1_extractive": round(
                robust_f1_extractive,
                4,
            ),
            "robust_f1_agreement": round(
                robust_f1_agreement,
                4,
            ),
            "robust_qc_level": robust_qc_level,
            "robust_approved": (
                robust_qc_level != "reject"
            ),
        }
    )

    stage4_robust_results.append(
        robust_result
    )


stage4_robust_df = pd.DataFrame(
    stage4_robust_results
)

print("ROBUST QC LEVEL DISTRIBUTION")

print(
    stage4_robust_df[
        "robust_qc_level"
    ].value_counts()
)

robust_approved_df = stage4_robust_df[
    stage4_robust_df["robust_approved"]
].copy()

print(
    "\nRobust approved pairs:",
    len(robust_approved_df),
)

print(
    "Robust approved unique answer spans:",
    robust_approved_df[
        "answer_span"
    ].nunique(),
)

display(
    stage4_robust_df[
        [
            "candidate_id",
            "question",
            "answer_span",
            "generative_qa_answer",
            "extractive_qa_answer",
            "extractive_confidence",
            "original_qc_level",
            "robust_f1_generative",
            "robust_f1_extractive",
            "robust_f1_agreement",
            "robust_qc_level",
        ]
    ]
)

ROBUST QC LEVEL DISTRIBUTION
robust_qc_level
strict    9
reject    8
Name: count, dtype: int64

Robust approved pairs: 9
Robust approved unique answer spans: 5


,candidate_id,question,answer_span,generative_qa_answer,extractive_qa_answer,extractive_confidence,original_qc_level,robust_f1_generative,robust_f1_extractive,robust_f1_agreement,robust_qc_level
0,Q001,Who was the owner of the wallet?,Emily,Mr. Johnson.,Mr. Johnson,0.4695,reject,0.0000,0.0,1.0000,reject
1,Q002,Who was the owner of the lost wallet?,Emily,Mr. Johnson.,Mr. Johnson,0.6055,reject,0.0000,0.0,1.0000,reject
2,Q003,Where did Emily and Timmy meet Mr. Johnson?,the park,the nearby community centre .,community centre,0.4044,reject,0.0000,0.0,1.0000,reject
3,Q004,Where did the man walk through earlier that day?,the park,the park .,the park,0.4009,strict,1.0000,1.0,1.0000,strict
4,Q005,Where did Emily and Timmy meet to find the owner of the wallet?,the park,at the nearby community centre .,community centre,0.4275,reject,0.0000,0.0,1.0000,reject
5,Q006,Who did Emily ask to help her find the owner of the wallet?,Timmy,her friend Timmy.,Timmy,0.5408,reject,1.0000,1.0,1.0000,strict
6,Q007,What did Emily find on the ground?,a small leather wallet,"a small leather wallet. Inside were some money, a business card and an identification card.",a small leather wallet,0.4663,reject,1.0000,1.0,1.0000,strict
7,Q008,What did Emily discover on the ground?,a small leather wallet,"a small leather wallet. Inside were some money, a business card and an identification card.",a small leather wallet,0.5151,reject,1.0000,1.0,1.0000,strict
8,Q009,What did Emily find on a sunny day in the park?,a small leather wallet,"a small leather wallet. Inside were some money, a business card and an identification card.",a small leather wallet,0.4743,reject,1.0000,1.0,1.0000,strict
9,Q010,What did Emily and Timmy find on the business card?,the number,a telephone number and an identification card .,a telephone number,0.5996,reject,0.3333,1.0,1.0000,reject


In [12]:
QC_PRIORITY = {
    "strict": 3,
    "medium": 2,
    "mild": 1,
    "reject": 0,
}

approved_candidates = stage4_robust_df[
    stage4_robust_df["robust_approved"]
].copy()

approved_candidates["qc_priority"] = (
    approved_candidates[
        "robust_qc_level"
    ].map(QC_PRIORITY)
)

approved_candidates["verification_score"] = (
    approved_candidates[
        [
            "robust_f1_generative",
            "robust_f1_extractive",
            "robust_f1_agreement",
        ]
    ].mean(axis=1)
)

# Prefer:
# 1. Stronger QC level
# 2. Higher independent-model confidence
# 3. Higher average verification score
# 4. Higher Stage 3 question-quality score
approved_candidates = approved_candidates.sort_values(
    by=[
        "answer_span",
        "qc_priority",
        "extractive_confidence",
        "verification_score",
        "question_quality_score",
    ],
    ascending=[
        True,
        False,
        False,
        False,
        False,
    ],
)

final_verified_pairs_df = (
    approved_candidates
    .drop_duplicates(
        subset=["answer_span"],
        keep="first",
    )
    .copy()
)

final_verified_pairs_df = (
    final_verified_pairs_df
    .sort_values(
        by=[
            "story_section",
            "support_sentence_index",
        ]
    )
    .reset_index(drop=True)
)

final_verified_pairs_df.insert(
    0,
    "final_pair_id",
    [
        f"QA{index:02d}"
        for index in range(
            1,
            len(final_verified_pairs_df) + 1,
        )
    ],
)

print(
    "Final unique verified pairs:",
    len(final_verified_pairs_df),
)

display(
    final_verified_pairs_df[
        [
            "final_pair_id",
            "question",
            "answer_span",
            "extractive_confidence",
            "verification_score",
            "robust_qc_level",
            "story_section",
        ]
    ]
)

Final unique verified pairs: 5


,final_pair_id,question,answer_span,extractive_confidence,verification_score,robust_qc_level,story_section
0,QA01,Where did the man walk through earlier that day?,the park,0.4009,1.000000,strict,beginning
1,QA02,What did Emily discover on the ground?,a small leather wallet,0.5151,1.000000,strict,beginning
2,QA03,Who did Emily ask to help her find the owner of the wallet?,Timmy,0.5408,1.000000,strict,beginning
3,QA04,What did Emily feel proud of?,an honest and responsible decision,0.3915,0.944433,strict,ending
4,QA05,What did Emily and Timmy look for on the business card?,the number,0.6462,1.000000,strict,middle


In [13]:
ANSWER_DISPLAY_OVERRIDES = {
    "number": "a telephone number",
}

final_verified_pairs_df[
    "final_display_answer"
] = final_verified_pairs_df[
    "answer_span"
].apply(
    lambda answer: (
        ANSWER_DISPLAY_OVERRIDES.get(
            normalize_answer_for_scoring(
                answer
            ),
            answer,
        )
    )
)

print("FINAL VERIFIED QA PAIRS")
print("=" * 80)

for _, row in final_verified_pairs_df.iterrows():

    print(
        f"{row['final_pair_id']}. "
        f"{row['question']}"
    )

    print(
        "   Answer:",
        row["final_display_answer"],
    )

    print(
        "   QC:",
        row["robust_qc_level"],
    )

    print(
        "   Extractive confidence:",
        row["extractive_confidence"],
    )

    print()

FINAL VERIFIED QA PAIRS
QA01. Where did the man walk through earlier that day?
   Answer: the park
   QC: strict
   Extractive confidence: 0.4009

QA02. What did Emily discover on the ground?
   Answer: a small leather wallet
   QC: strict
   Extractive confidence: 0.5151

QA03. Who did Emily ask to help her find the owner of the wallet?
   Answer: Timmy
   QC: strict
   Extractive confidence: 0.5408

QA04. What did Emily feel proud of?
   Answer: an honest and responsible decision
   QC: strict
   Extractive confidence: 0.3915

QA05. What did Emily and Timmy look for on the business card?
   Answer: a telephone number
   QC: strict
   Extractive confidence: 0.6462



In [14]:
STORY_SECTION_ORDER = {
    "beginning": 0,
    "middle": 1,
    "ending": 2,
}

final_verified_pairs_df[
    "section_order"
] = final_verified_pairs_df[
    "story_section"
].map(STORY_SECTION_ORDER)

final_verified_pairs_df = (
    final_verified_pairs_df
    .sort_values(
        by=[
            "section_order",
            "support_sentence_index",
        ],
        ascending=True,
    )
    .reset_index(drop=True)
)

final_verified_pairs_df[
    "final_pair_id"
] = [
    f"QA{index:02d}"
    for index in range(
        1,
        len(final_verified_pairs_df) + 1,
    )
]

ANSWER_DISPLAY_OVERRIDES = {
    "number": "a telephone number",
}

final_verified_pairs_df[
    "final_display_answer"
] = final_verified_pairs_df[
    "answer_span"
].apply(
    lambda answer: (
        ANSWER_DISPLAY_OVERRIDES.get(
            normalize_answer_for_scoring(answer),
            answer,
        )
    )
)

print("FINAL VERIFIED QA PAIRS IN STORY ORDER")
print("=" * 80)

for _, row in final_verified_pairs_df.iterrows():

    print(
        f"{row['final_pair_id']}. "
        f"{row['question']}"
    )

    print(
        "   Answer:",
        row["final_display_answer"],
    )

    print(
        "   Section:",
        row["story_section"],
    )

    print(
        "   QC:",
        row["robust_qc_level"],
    )

    print()

FINAL VERIFIED QA PAIRS IN STORY ORDER
QA01. Where did the man walk through earlier that day?
   Answer: the park
   Section: beginning
   QC: strict

QA02. What did Emily discover on the ground?
   Answer: a small leather wallet
   Section: beginning
   QC: strict

QA03. Who did Emily ask to help her find the owner of the wallet?
   Answer: Timmy
   Section: beginning
   QC: strict

QA04. What did Emily and Timmy look for on the business card?
   Answer: a telephone number
   Section: middle
   QC: strict

QA05. What did Emily feel proud of?
   Answer: an honest and responsible decision
   Section: ending
   QC: strict



In [15]:
from datetime import datetime, timezone

FINAL_MANUAL_APPROVAL = True

FINAL_REVIEW_NOTES = [
    "Verified questions are answerable from the story.",
    "Incorrect owner and location questions were rejected.",
    "One best question was retained for each answer span.",
    "Final questions were arranged in story order.",
]

STAGE4_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/KidStory-Qwen2.5/"
    "learnguard_stage4_results"
)

STAGE4_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

safe_topic = re.sub(
    r"[^a-z0-9]+",
    "_",
    TOPIC.lower(),
).strip("_")

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

stage4_output_path = STAGE4_OUTPUT_DIR / (
    f"{timestamp}_{safe_topic}_verified_qa_pairs.json"
)

final_pairs = (
    final_verified_pairs_df
    .drop(columns=["section_order"], errors="ignore")
    .to_dict(orient="records")
)

stage4_record = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "stage4_status": "COMPLETE",

    "source_stage3_file": str(STAGE3_JSON),

    "qa_model_path": str(QA_MODEL_PATH),

    "independent_verifier": (
        "deepset/roberta-base-squad2"
    ),

    "topic": TOPIC,
    "age_group": AGE_GROUP,
    "story": STORY,

    "candidate_count": len(
        stage4_robust_results
    ),

    "robust_qc_distribution": (
        stage4_robust_df[
            "robust_qc_level"
        ]
        .value_counts()
        .to_dict()
    ),

    "all_verified_candidates": (
        stage4_robust_results
    ),

    "final_verified_qa_pairs": final_pairs,

    "final_pair_count": len(final_pairs),

    "manual_review": {
        "approved": FINAL_MANUAL_APPROVAL,
        "notes": FINAL_REVIEW_NOTES,
    },
}

with stage4_output_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        stage4_record,
        file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

print("Stage 4 result saved:")
print(stage4_output_path)
print()
print(
    "Stage 4 status:",
    stage4_record["stage4_status"],
)
print(
    "Final verified QA pairs:",
    stage4_record["final_pair_count"],
)
print(
    "Manual approval:",
    stage4_record[
        "manual_review"
    ]["approved"],
)

Stage 4 result saved:
/content/drive/MyDrive/KidStory-Qwen2.5/learnguard_stage4_results/20260802_052906_honesty_when_a_child_finds_a_lost_wallet_verified_qa_pairs.json

Stage 4 status: COMPLETE
Final verified QA pairs: 5
Manual approval: True
